# OULAD Student Performance — Full Model Comparison

**Models:** XGBoost · LightGBM · CatBoost · Random Forest · Extra Trees · Bagged DT · Decision Tree · MLP · DNN (PyTorch GPU/CPU)

**Tasks:** Binary (AtRisk vs Success) and 4-Class (Distinction/Fail/Pass/Withdrawn)

> All features are **end-of-course** (final-outcome prediction). For early-warning use the Week-8 table.

---

## 1. Install dependencies

In [ ]:
!pip install -q xgboost lightgbm catboost shap optuna scikit-learn pandas numpy matplotlib seaborn
# PyTorch — GPU build if available, falls back to CPU automatically
import subprocess, sys
try:
    import torch
    print(f'PyTorch already installed: {torch.__version__}  CUDA={torch.cuda.is_available()}')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
print('All packages ready.')

## 2. Upload or mount the OULAD ML table

Set `USE_DRIVE = True` if `oulad_ml_table.csv` is in your Google Drive. Otherwise the cell will prompt you to upload it directly.

In [ ]:
import os
from pathlib import Path

USE_DRIVE  = False   # ← set True if using Google Drive
DRIVE_PATH = 'MyDrive/student_data/oulad_ml_table.csv'  # adjust if needed

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path('/content/drive') / DRIVE_PATH
else:
    from google.colab import files as _f
    print('Upload oulad_ml_table.csv:')
    up = _f.upload()
    DATA_PATH = Path(list(up.keys())[0])

assert DATA_PATH.exists(), f'File not found: {DATA_PATH}'
print(f'Dataset: {DATA_PATH}  ({DATA_PATH.stat().st_size/1e6:.1f} MB)')

## 3. Imports & global config

In [ ]:
import warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, OrdinalEncoder, LabelEncoder, label_binarize)
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    cohen_kappa_score, matthews_corrcoef,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc, precision_recall_curve)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import shap
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  PyTorch {torch.__version__}')

matplotlib.rcParams.update({'font.size': 11, 'axes.titlesize': 12,
    'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9})

RESULTS_DIR = Path('/content/results')
FIGS_DIR    = Path('/content/figures')
RESULTS_DIR.mkdir(exist_ok=True)
FIGS_DIR.mkdir(exist_ok=True)
print('Imports OK.')

## 4. OULAD feature engineering

Reproduces the same engineering used in `high_accuracy_pipeline.py` and `extended_models_pipeline.py`.

In [ ]:
OULAD_DROP = {
    'date_unregistration','date_unreg','date_unregistered','weighted_score',
    'active_weeks','clicks_per_active_week','assessments_per_week',
    'activity_count','days_active','avg_clicks_per_day',
    'registration_delay_category','id_student','id_assessment','id_site',
    'first_ts','last_ts','last_assessment_day','first_assessment_day',
    'code_module','code_presentation',
}
BINARY_MAP = {'Pass':'Success','Distinction':'Success','Fail':'AtRisk','Withdrawn':'AtRisk'}

def engineer_oulad(df: pd.DataFrame, binary: bool = False) -> pd.DataFrame:
    df = df.copy()
    df = df.drop(columns=[c for c in OULAD_DROP if c in df.columns])
    zero_fill = (
        [f'week{w}_clicks' for w in range(1,13)]
        + ['total_clicks','avg_score','score_std','num_assessments',
           'assessment_completion_ratio','missed_assessments',
           'late_submission_count','assessment_score_trend',
           'click_variance','click_growth_rate','longest_inactive_gap',
           'week_click_sum_1_12','inactivity_days',
           'clicks_until_week2','clicks_until_week4',
           'clicks_until_week6','clicks_until_week8']
        + [c for c in df.columns if 'activity_type_' in c]
    )
    for col in zero_fill:
        if col in df.columns: df[col] = df[col].fillna(0)
    for col in ['total_clicks']+[f'week{w}_clicks' for w in range(1,13)]:
        if col in df.columns: df[f'log_{col}'] = np.log1p(df[col])
    if 'clicks_until_week4' in df.columns and 'total_clicks' in df.columns:
        df['early_engagement_ratio'] = (df['clicks_until_week4']
            / df['total_clicks'].replace(0,np.nan)).fillna(0).clip(0,1)
    if 'clicks_until_week8' in df.columns and 'total_clicks' in df.columns:
        df['mid_engagement_ratio'] = (df['clicks_until_week8']
            / df['total_clicks'].replace(0,np.nan)).fillna(0).clip(0,1)
    if 'avg_score' in df.columns and 'studied_credits' in df.columns:
        df['score_per_credit'] = (df['avg_score']
            / df['studied_credits'].replace(0,np.nan)).fillna(0)
    if 'avg_score' in df.columns and 'assessment_completion_ratio' in df.columns:
        df['score_x_completion'] = df['avg_score'] * df['assessment_completion_ratio']
    if 'avg_score' in df.columns:
        df['passed_assessments'] = (df['avg_score'] >= 40).astype(int)
    if 'assessment_completion_ratio' in df.columns:
        df['submitted_all'] = (df['assessment_completion_ratio'] >= 1.0).astype(int)
    if 'imd_band' in df.columns:
        imd_order = {'0-10%':1,'10-20':2,'10-20%':2,'20-30%':3,'30-40%':4,
                     '40-50%':5,'50-60%':6,'60-70%':7,'70-80%':8,'80-90%':9,'90-100%':10}
        df['imd_numeric'] = df['imd_band'].map(imd_order).fillna(5)
    if 'total_clicks' in df.columns:
        df['zero_clicks'] = (df['total_clicks']==0).astype(int)
    if 'num_of_prev_attempts' in df.columns:
        df['is_repeat_student'] = (df['num_of_prev_attempts']>0).astype(int)
    week_cols = [f'week{w}_clicks' for w in range(1,13) if f'week{w}_clicks' in df.columns]
    if len(week_cols) >= 4:
        df['peak_week_clicks'] = df[week_cols].max(axis=1)
        df['active_weeks_count'] = (df[week_cols]>0).sum(axis=1)
        h1 = [f'week{w}_clicks' for w in range(1,7)  if f'week{w}_clicks' in df.columns]
        h2 = [f'week{w}_clicks' for w in range(7,13) if f'week{w}_clicks' in df.columns]
        if h1 and h2:
            df['h2_vs_h1_ratio'] = df[h2].sum(axis=1) / (df[h1].sum(axis=1)+1)
    if binary and 'final_result' in df.columns:
        df['final_result'] = df['final_result'].map(BINARY_MAP).fillna('AtRisk')
    return df

print('Feature engineering function defined.')

## 5. Load and prepare data

In [ ]:
def load_oulad(binary: bool = False):
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df.columns = df.columns.str.strip()
    assert 'final_result' in df.columns, 'final_result column not found'
    df = engineer_oulad(df, binary=binary)
    df = df.dropna(subset=['final_result'])
    y = df['final_result'].astype(str)
    X = df.drop(columns=['final_result'], errors='ignore').dropna(axis=1, how='all')
    print(f'  Loaded {'binary' if binary else '4-class'}: '
          f'shape={X.shape}  classes={y.value_counts().to_dict()}')
    return X, y

X_bin, y_bin_raw = load_oulad(binary=True)
X_4cl, y_4cl_raw = load_oulad(binary=False)
print('Data loaded.')

## 6. Preprocessing pipelines

In [ ]:
def make_preprocessor(X, scale=True):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
    parts = []
    if num_cols:
        num_steps = [('imp', SimpleImputer(strategy='median'))]
        if scale: num_steps.append(('scl', StandardScaler()))
        parts.append(('num', Pipeline(num_steps), num_cols))
    if cat_cols:
        if scale:
            cat_steps = [('imp', SimpleImputer(strategy='most_frequent')),
                         ('ohe', OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False, max_categories=30))]
        else:
            cat_steps = [('imp', SimpleImputer(strategy='most_frequent')),
                         ('ord', OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1))]
        parts.append(('cat', Pipeline(cat_steps), cat_cols))
    return ColumnTransformer(parts, remainder='drop')

print('Preprocessor factory ready.')

## 7. PyTorch DNN (GPU/CPU)

In [ ]:
class _Net(nn.Module):
    def __init__(self, in_dim, n_cls, hidden=(512,256,128), drop=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(drop)]
            prev = h
        layers.append(nn.Linear(prev, n_cls))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class DNNClassifier:
    """sklearn-compatible wrapper; uses GPU if available."""
    def __init__(self, n_classes=2, epochs=60, bs=512, lr=1e-3,
                 hidden=(512,256,128), drop=0.3, seed=SEED):
        self.n_classes=n_classes; self.epochs=epochs; self.bs=bs
        self.lr=lr; self.hidden=hidden; self.drop=drop; self.seed=seed
        self._m=None; self.classes_=None

    def fit(self, X, y):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        self.classes_ = np.unique(y)
        X_t = torch.tensor(X.astype(np.float32), device=DEVICE)
        y_t = torch.tensor(y.astype(np.int64),   device=DEVICE)
        counts  = np.bincount(y); w = 1./(counts+1e-8)
        w = torch.tensor(w/w.sum()*len(counts), dtype=torch.float32, device=DEVICE)
        self._m = _Net(X_t.shape[1], len(self.classes_),
                       self.hidden, self.drop).to(DEVICE)
        opt = torch.optim.AdamW(self._m.parameters(), lr=self.lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)
        crit = nn.CrossEntropyLoss(weight=w)
        ds  = TensorDataset(X_t, y_t)
        ldr = DataLoader(ds, batch_size=self.bs, shuffle=True)
        self._m.train()
        for _ in range(self.epochs):
            for xb,yb in ldr:
                opt.zero_grad(); loss=crit(self._m(xb),yb); loss.backward(); opt.step()
            sch.step()
        return self

    def predict_proba(self, X):
        self._m.eval()
        with torch.no_grad():
            logits = self._m(torch.tensor(X.astype(np.float32), device=DEVICE))
            return torch.softmax(logits,1).cpu().numpy()

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

print('DNN defined  (device:', DEVICE, ')')

## 8. Model registry (all 9 models)

In [ ]:
def get_all_models(n_classes, seed=SEED):
    """
    Returns: {model_key: (classifier, needs_scaling)}
    needs_scaling=True  -> StandardScaler applied (MLP, DNN, LogReg)
    needs_scaling=False -> OrdinalEncoder only   (tree-based models)
    """
    cw = 'balanced'
    return {
        'xgboost': (XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85, gamma=0,
            min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
            eval_metric='mlogloss', random_state=seed,
            n_jobs=1, verbosity=0, tree_method='hist'), False),
        'lightgbm': (LGBMClassifier(
            n_estimators=500, num_leaves=63, learning_rate=0.05,
            feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
            min_child_samples=20, reg_alpha=0.1, reg_lambda=0.5,
            class_weight=cw, random_state=seed, n_jobs=1, verbosity=-1), False),
        'catboost': (CatBoostClassifier(
            iterations=500, depth=7, learning_rate=0.05,
            l2_leaf_reg=3, verbose=0, random_state=seed,
            auto_class_weights='Balanced'), False),
        'random_forest': (RandomForestClassifier(
            n_estimators=400, min_samples_leaf=2,
            class_weight=cw, random_state=seed, n_jobs=1), False),
        'et': (ExtraTreesClassifier(
            n_estimators=500, min_samples_leaf=2,
            class_weight=cw, random_state=seed, n_jobs=1), False),
        'bdt': (BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=8,
                class_weight=cw, random_state=seed),
            n_estimators=200, max_samples=0.8, max_features=0.8,
            bootstrap=True, random_state=seed, n_jobs=1), False),
        'dt': (DecisionTreeClassifier(
            max_depth=12, min_samples_leaf=5,
            class_weight=cw, random_state=seed), False),
        'mlp': (MLPClassifier(
            hidden_layer_sizes=(512,256,128), activation='relu',
            solver='adam', alpha=1e-4, batch_size=256,
            learning_rate='adaptive', max_iter=300,
            early_stopping=True, validation_fraction=0.1,
            random_state=seed), True),
        'dnn': (DNNClassifier(n_classes=n_classes, epochs=60, seed=seed), True),
    }

MODEL_LABELS = {
    'xgboost':'XGBoost', 'lightgbm':'LightGBM', 'catboost':'CatBoost',
    'random_forest':'Random Forest', 'et':'Extra Trees', 'bdt':'Bagged DT',
    'dt':'Decision Tree', 'mlp':'MLP', 'dnn':'DNN (PyTorch)',
}
PALETTE = ['#4878D0','#EE854A','#6ACC65','#D65F5F',
           '#B47CC7','#C4AD66','#77BEDB','#E47298','#70A89F']
MODEL_ORDER = list(MODEL_LABELS.keys())
COLORS = {k: PALETTE[i] for i,k in enumerate(MODEL_ORDER)}
print('Model registry ready:', MODEL_ORDER)

## 9. Evaluation helper

In [ ]:
def evaluate(y_true, y_pred, y_prob, le, model_name, ds_name):
    r = {
        'dataset': ds_name, 'model': model_name,
        'n_samples': len(y_true), 'n_classes': len(np.unique(y_true)),
        'accuracy':          accuracy_score(y_true, y_pred),
        'f1_macro':          f1_score(y_true, y_pred, average='macro',    zero_division=0),
        'f1_weighted':       f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'cohen_kappa':       cohen_kappa_score(y_true, y_pred),
        'mcc':               matthews_corrcoef(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            if len(le.classes_) == 2:
                r['roc_auc'] = roc_auc_score(y_true, y_prob[:,1])
            else:
                yb = label_binarize(y_true, classes=np.arange(len(le.classes_)))
                r['roc_auc'] = roc_auc_score(yb, y_prob, average='macro',
                                              multi_class='ovr')
        except Exception: r['roc_auc'] = float('nan')
    else:
        r['roc_auc'] = float('nan')
    return r

print('Evaluation helper ready.')

## 10. Training loop

Runs all 9 models on binary **and** 4-class tasks. Uses a single shared 80/20 stratified split per task so results are directly comparable.

In [ ]:
# Models to run — remove any you want to skip
MODELS_TO_RUN = ['xgboost','lightgbm','catboost','random_forest',
                 'et','bdt','dt','mlp','dnn']

all_results = []
all_preds   = {}   # {(mode, model_key): pred_df}
all_shap    = {}   # {(mode, model_key): shap_df}

for binary, (X_data, y_raw) in [(True,(X_bin,y_bin_raw)), (False,(X_4cl,y_4cl_raw))]:
    mode = 'binary' if binary else '4class'
    label = f'oulad_{mode}'
    print(f'\n{"="*65}')
    print(f'  {label.upper()}')
    print(f'{"="*65}')

    le = LabelEncoder()
    y_enc = le.fit_transform(y_raw)
    n_cls = len(le.classes_)
    num_cols = X_data.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_data.select_dtypes(include=['object','category']).columns.tolist()

    # Single shared split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_data, pd.Series(y_enc, index=X_data.index),
        test_size=0.2, random_state=SEED, stratify=y_enc)
    print(f'  Split: train={len(X_tr)}  test={len(X_te)}  classes={list(le.classes_)}')

    models = get_all_models(n_cls)

    for mk in MODELS_TO_RUN:
        clf, needs_scale = models[mk]
        pre = make_preprocessor(X_data, scale=needs_scale)
        t0 = time.time()
        print(f'\n  [{mk}] fitting...')

        X_tr_t = pre.fit_transform(X_tr)
        X_te_t = pre.transform(X_te)
        clf.fit(X_tr_t, y_tr.values)

        y_pred = np.array(clf.predict(X_te_t))
        try: y_prob = clf.predict_proba(X_te_t)
        except: y_prob = None

        # Decode preds
        y_pred_i = y_pred.astype(int) if y_pred.dtype.kind in 'iu' else y_pred
        y_te_str   = le.inverse_transform(y_te.values.astype(int))
        y_pred_str = le.inverse_transform(y_pred_i)

        r = evaluate(y_te.values, y_pred_i, y_prob, le, mk, label)
        r['time_s'] = round(time.time()-t0, 1)
        all_results.append(r)

        # Store predictions
        pdf = pd.DataFrame({'y_true': y_te_str, 'y_pred': y_pred_str})
        if y_prob is not None:
            for ci, cn in enumerate(le.classes_):
                pdf[f'prob_{cn}'] = y_prob[:, ci]
            pdf['confidence'] = y_prob.max(axis=1)
        all_preds[(mode, mk)] = pdf
        pdf.to_csv(RESULTS_DIR/f'predictions_{label}_{mk}.csv', index=False)

        print(f'  [{mk}] acc={r["accuracy"]:.4f}  f1={r["f1_macro"]:.4f}  '
              f'bacc={r["balanced_accuracy"]:.4f}  roc={r["roc_auc"]:.4f}  '
              f'({r["time_s"]}s)')

results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_DIR/'all_models_results.csv', index=False)
print('\nAll models trained. Results saved.')

## 11. Summary results table

In [ ]:
cols_show = ['dataset','model','accuracy','f1_macro','balanced_accuracy',
             'cohen_kappa','mcc','roc_auc','time_s']
display_df = results_df[cols_show].sort_values(['dataset','accuracy'], ascending=[True,False])
display_df = display_df.reset_index(drop=True)

# Style the table
def highlight_best(s):
    is_best = s == s.max()
    return ['background-color: #d4edda; font-weight: bold' if v else '' for v in is_best]

for ds in display_df['dataset'].unique():
    sub = display_df[display_df['dataset']==ds].copy()
    print(f'\n=== {ds} ===')
    print(sub.drop(columns=['dataset'])
          .set_index('model')
          .to_string(float_format='{:.4f}'.format))

## 12. Performance bar charts — Binary & 4-Class

In [ ]:
def plot_perf_bars(mode, title_suffix):
    sub = results_df[results_df['dataset']==f'oulad_{mode}'].copy()
    sub['model_key'] = sub['model']
    keys = [k for k in MODEL_ORDER if k in sub['model_key'].values]
    metrics = ['accuracy','f1_macro','balanced_accuracy','roc_auc']
    labels  = ['Accuracy','F1 Macro','Balanced Acc','ROC-AUC']
    x = np.arange(len(metrics))
    n = len(keys); w = 0.8/max(n,1)
    offsets = np.linspace(-(n-1)/2,(n-1)/2,n)*w
    fig, ax = plt.subplots(figsize=(max(12,n*1.4), 5))
    for i, mk in enumerate(keys):
        row = sub[sub['model_key']==mk].iloc[0]
        vals = [row.get(c, np.nan) for c in metrics]
        bars = ax.bar(x+offsets[i], vals, w,
                      label=MODEL_LABELS.get(mk,mk),
                      color=COLORS.get(mk,'#888'), alpha=0.88)
        for bar,v in zip(bars,vals):
            if pd.notna(v):
                ax.text(bar.get_x()+bar.get_width()/2, float(v)+0.003,
                        f'{float(v):.3f}', ha='center', va='bottom', fontsize=6.5)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ylim_bot = 0.85 if mode=='binary' else 0.55
    ax.set_ylim(ylim_bot, 1.01); ax.set_ylabel('Score')
    ax.set_title(f'{title_suffix} — All Models Performance')
    ax.legend(loc='lower right', fontsize=8, ncol=3)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGS_DIR/f'perf_bars_{mode}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_perf_bars('binary',  'Binary (AtRisk vs Success)')
plot_perf_bars('4class',  '4-Class (Distinction/Fail/Pass/Withdrawn)')

## 13. ROC curves — all models

In [ ]:
def plot_roc(mode, classes_list):
    fig, ax = plt.subplots(figsize=(9, 7))
    for mk in MODEL_ORDER:
        pdf = all_preds.get((mode, mk))
        if pdf is None: continue
        if mode == 'binary':
            if 'prob_AtRisk' not in pdf.columns: continue
            y_bin = (pdf['y_true']=='AtRisk').astype(int)
            fpr, tpr, _ = roc_curve(y_bin, pdf['prob_AtRisk'])
            roc_auc_val = auc(fpr, tpr)
            ax.plot(fpr, tpr, lw=2, color=COLORS.get(mk,'#888'),
                    label=f'{MODEL_LABELS.get(mk,mk):<22} AUC={roc_auc_val:.4f}')
        else:
            prob_cols = [c for c in pdf.columns if c.startswith('prob_')]
            if not prob_cols: continue
            yb = label_binarize(pdf['y_true'], classes=classes_list)
            pm = pdf[prob_cols].values
            all_fpr = np.unique(np.concatenate(
                [roc_curve(yb[:,k], pm[:,k])[0] for k in range(len(classes_list))]))
            mt = np.zeros_like(all_fpr)
            for k in range(len(classes_list)):
                fp,tp,_ = roc_curve(yb[:,k], pm[:,k])
                mt += np.interp(all_fpr, fp, tp)
            mt /= len(classes_list)
            mac_auc = auc(all_fpr, mt)
            ax.plot(all_fpr, mt, lw=2, color=COLORS.get(mk,'#888'),
                    label=f'{MODEL_LABELS.get(mk,mk):<22} Macro AUC={mac_auc:.4f}')
    ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
    ax.set_xlim([-0.01,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'{mode.upper()} ROC Curves — All Models')
    ax.legend(loc='lower right', fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGS_DIR/f'roc_{mode}.png', dpi=150, bbox_inches='tight')
    plt.show()

FOUR_CLASSES = ['Distinction','Fail','Pass','Withdrawn']
plot_roc('binary',  ['AtRisk','Success'])
plot_roc('4class',  FOUR_CLASSES)

## 14. Confusion matrices — all models

In [ ]:
def plot_cm_grid(mode, label_order):
    keys = [mk for mk in MODEL_ORDER if all_preds.get((mode,mk)) is not None]
    ncols = 5; nrows = (len(keys)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 4.2*nrows))
    axes = np.array(axes).reshape(-1)
    fig.suptitle(f'{mode.upper()} Confusion Matrices — All Models', fontsize=13, y=1.01)
    for i, mk in enumerate(keys):
        ax = axes[i]
        pdf = all_preds[(mode,mk)]
        labels = label_order if label_order else sorted(pdf['y_true'].unique())
        cm = confusion_matrix(pdf['y_true'], pdf['y_pred'], labels=labels)
        ConfusionMatrixDisplay(cm, display_labels=labels).plot(
            ax=ax, values_format='d', colorbar=False, cmap='Blues')
        ax.set_title(f'{MODEL_LABELS.get(mk,mk)}', fontsize=10)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual' if i%ncols==0 else '')
        if mode=='4class': ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    for j in range(len(keys), len(axes)): axes[j].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIGS_DIR/f'cm_{mode}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_cm_grid('binary',  ['AtRisk','Success'])
plot_cm_grid('4class',  FOUR_CLASSES)

## 15. Precision-Recall curves — binary

In [ ]:
fig, ax = plt.subplots(figsize=(9,7))
for mk in MODEL_ORDER:
    pdf = all_preds.get(('binary', mk))
    if pdf is None or 'prob_AtRisk' not in pdf.columns: continue
    y_bin = (pdf['y_true']=='AtRisk').astype(int)
    p, r, _ = precision_recall_curve(y_bin, pdf['prob_AtRisk'])
    pr_auc = auc(r, p)
    ax.plot(r, p, lw=2, color=COLORS.get(mk,'#888'),
            label=f'{MODEL_LABELS.get(mk,mk):<22} AUC={pr_auc:.4f}')
baseline = (all_preds[('binary', list(all_preds.keys())[0])]['y_true']=='AtRisk').mean()
ax.axhline(baseline, color='k', ls='--', lw=1, label=f'Baseline (prev={baseline:.2f})')
ax.set_xlim([0,1.01]); ax.set_ylim([0,1.02])
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Binary Precision-Recall Curves — All Models')
ax.legend(loc='lower left', fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR/'pr_binary.png', dpi=150, bbox_inches='tight')
plt.show()

## 16. SHAP feature importance

SHAP TreeExplainer for tree-based models (fast). KernelExplainer for MLP/DNN (slow — uses 100 background samples).

In [ ]:
TREE_MODELS = {'xgboost','lightgbm','catboost','random_forest','et','bdt','dt'}

def compute_shap_importance(clf, X_sample, feat_names, mk):
    try:
        if mk in TREE_MODELS:
            exp = shap.TreeExplainer(clf)
            sv  = exp.shap_values(X_sample)
        else:
            bg  = shap.sample(X_sample, min(50, len(X_sample)))
            exp = shap.KernelExplainer(clf.predict_proba, bg)
            sv  = exp.shap_values(X_sample[:100], nsamples=50)
        if isinstance(sv, list): vals = np.mean([np.abs(v) for v in sv],0).mean(0)
        elif sv.ndim==3:         vals = np.abs(sv).mean(axis=(0,2))
        else:                    vals = np.abs(sv).mean(0)
        n = min(len(feat_names), len(vals))
        order = np.argsort(vals[:n])[::-1][:20]
        return pd.DataFrame({'feature':[feat_names[i] for i in order],
                             'shap_importance':[float(vals[i]) for i in order]})
    except Exception as e:
        print(f'    SHAP failed for {mk}: {e}')
        return None

print('SHAP helper defined. Run next cell to compute SHAP for tree models.')

In [ ]:
# Compute SHAP for tree-based models (binary task)
# Remove models from shap_models if you want to skip them
shap_models = ['xgboost','lightgbm','catboost','random_forest','et']

# Re-fit preprocessors to get feature names (transformers not stored from training loop)
le_b = LabelEncoder().fit(y_bin_raw)
y_enc_b = le_b.transform(y_bin_raw)
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_bin, pd.Series(y_enc_b, index=X_bin.index),
    test_size=0.2, random_state=SEED, stratify=y_enc_b)

for mk in shap_models:
    print(f'  SHAP [{mk}]...')
    clf, needs_scale = get_all_models(2)[mk]
    pre = make_preprocessor(X_bin, scale=needs_scale)
    X_tr_t = pre.fit_transform(X_tr_b)
    X_te_t = pre.transform(X_te_b)
    clf.fit(X_tr_t, y_tr_b.values)
    try:
        feat_names = [n.replace('num__','').replace('cat__ohe__','')
                       .replace('cat__ord__','').replace('cat__','')
                      for n in pre.get_feature_names_out()]
    except: feat_names = [f'f{i}' for i in range(X_te_t.shape[1])]
    n_s = min(500, len(X_te_t))
    sd = compute_shap_importance(clf, X_te_t[:n_s], feat_names, mk)
    if sd is not None:
        all_shap[('binary', mk)] = sd
        sd.to_csv(RESULTS_DIR/f'shap_oulad_binary_{mk}.csv', index=False)
        print(f'    Top: {sd.head(3)["feature"].tolist()}')

print('SHAP done.')

## 17. SHAP importance bar chart grid

In [ ]:
avail_shap = [(mk, all_shap[('binary',mk)]) for mk in MODEL_ORDER
              if ('binary',mk) in all_shap]
if avail_shap:
    n = len(avail_shap)
    ncols = 5; nrows = (n+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 6*nrows))
    axes = np.array(axes).reshape(-1)
    fig.suptitle('Binary — SHAP Feature Importance (All Models)', fontsize=13, y=1.01)
    for i, (mk, sd) in enumerate(avail_shap):
        ax = axes[i]
        top = sd.nlargest(15,'shap_importance').sort_values('shap_importance')
        ax.barh(top['feature'], top['shap_importance'],
                color=COLORS.get(mk,'#888'), alpha=0.85)
        ax.set_title(MODEL_LABELS.get(mk,mk), fontsize=10)
        ax.set_xlabel('Mean |SHAP|')
        if i%ncols==0: ax.set_ylabel('Feature')
        ax.grid(axis='x', alpha=0.3)
    for j in range(len(avail_shap),len(axes)): axes[j].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIGS_DIR/'shap_binary.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No SHAP data yet. Run cell 16 first.')

## 18. Performance heatmap

In [ ]:
for mode in ['binary','4class']:
    sub = results_df[results_df['dataset']==f'oulad_{mode}'].copy()
    sub = sub.set_index('model')[['accuracy','f1_macro','balanced_accuracy',
                                   'roc_auc','cohen_kappa','mcc']]
    # Reorder rows
    order = [k for k in MODEL_ORDER if k in sub.index]
    sub = sub.loc[order]
    sub.index = [MODEL_LABELS.get(k,k) for k in order]

    fig, ax = plt.subplots(figsize=(10, max(4, len(order)*0.5+2)))
    sns.heatmap(sub.astype(float), annot=True, fmt='.3f',
                cmap='YlGnBu', ax=ax, vmin=0.5, vmax=1.0,
                linewidths=0.5, cbar_kws={'shrink':0.7})
    ax.set_title(f'{mode.upper()} — Performance Heatmap (All Models)')
    ax.set_xlabel('')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIGS_DIR/f'heatmap_{mode}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 19. Training time comparison

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, mode in zip(axes, ['binary','4class']):
    sub = results_df[results_df['dataset']==f'oulad_{mode}'].copy()
    sub = sub[[c for c in sub.columns if c in ['model','time_s']]]
    if 'time_s' not in sub.columns: sub['time_s'] = 0
    sub = sub.sort_values('time_s', ascending=True)
    colors = [COLORS.get(m,'#888') for m in sub['model']]
    sub_labs = [MODEL_LABELS.get(m,m) for m in sub['model']]
    ax.barh(sub_labs, sub['time_s'], color=colors, alpha=0.85)
    ax.set_xlabel('Training time (s)')
    ax.set_title(f'{mode.upper()} — Training Time')
    ax.grid(axis='x', alpha=0.3)
    for i, v in enumerate(sub['time_s']):
        ax.text(v+0.2, i, f'{v:.1f}s', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(FIGS_DIR/'training_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 20. Download results

In [ ]:
import zipfile
zip_path = Path('/content/oulad_all_models_results.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in RESULTS_DIR.glob('*.csv'):
        zf.write(f, f.name)
    for f in FIGS_DIR.glob('*.png'):
        zf.write(f, f.name)

from google.colab import files
files.download(str(zip_path))
print(f'Downloaded: {zip_path.name}')